In [1]:
import pandas as pd
import os


In [2]:
all_files = os.listdir("Data/OutputData/FirstScrapeV2")
dfs = []
for file in all_files:
    if 'pickle' in file:
        print(file)
        df = pd.read_pickle(f"Data/OutputData/FirstScrapeV2/{file}")
        dfs.append(df)

all_dfs = pd.concat(dfs)
all_dfs.shape

Roofing_Companies_Alabama.pickle
Roofing_Companies_Arizona.pickle
Roofing_Companies_Delaware.pickle
Roofing_Companies_Georgia.pickle
Roofing_Companies_Kansas.pickle
Roofing_Companies_Louisiana.pickle
Roofing_Companies_Maryland.pickle
Roofing_Companies_Massachusetts.pickle
Roofing_Companies_Michigan.pickle
Roofing_Companies_Mississippi.pickle
Roofing_Companies_Missouri.pickle
Roofing_Companies_North Carolina.pickle
Roofing_Companies_Ohio.pickle
Roofing_Companies_Oklahoma.pickle
Roofing_Companies_Oregon .pickle
Roofing_Companies_Pennsylvania Indiana.pickle
Roofing_Companies_South Carolina.pickle
Roofing_Companies_Tennessee.pickle
Roofing_Companies_Texas.pickle
Roofing_Companies_Virginia.pickle
Roofing_Companies_Washington .pickle


(939, 4)

In [3]:
all_dfs

,search_google,all_text_first_page,google_map_link,PHYSICAL STATE
0,"Roofing companies near , Alabama, USA","Yellowhammer Roofing, Inc.\n4.8(1,179)\nRoofin...",https://www.google.com/maps/place/Yellowhammer...,Alabama
1,"Roofing companies near , Alabama, USA",Alabama Roofing Professionals\n4.9(308)\nRoofi...,https://www.google.com/maps/place/Alabama+Roof...,Alabama
2,"Roofing companies near , Alabama, USA",Advantage Plus Roofing LLC: Alabama Roofer\n4....,https://www.google.com/maps/place/Advantage+Pl...,Alabama
3,"Roofing companies near , Alabama, USA",Blue Angels Roofing\n4.8(416)\nRoofing contrac...,https://www.google.com/maps/place/Blue+Angels+...,Alabama
4,"Roofing companies near , Alabama, USA","Advanced Roofing & Construction, LLC\n5.0(700)...",https://www.google.com/maps/place/Advanced+Roo...,Alabama
...,...,...,...,...
25,"Roofing companies near , Washington , USA",Sunshine Woodinville Roof Repair\n4.9(85)\nRoo...,https://www.google.com/maps/place/Sunshine+Woo...,Washington
26,"Roofing companies near , Washington , USA",Water Wise Roof Service Llc\n5.0(12)\nRoofing ...,https://www.google.com/maps/place/Water+Wise+R...,Washington
27,"Roofing companies near , Washington , USA",V & R Roofing\n4.9(10)\nRoofing contractor · ...,https://www.google.com/maps/place/V+%26+R+Roof...,Washington
28,"Roofing companies near , Washington , USA",Robert Daniel Roofing\n5.0(36)\nRoofing contra...,https://www.google.com/maps/place/Robert+Danie...,Washington


In [5]:
main_data = all_dfs.drop_duplicates(subset='google_map_link')

In [4]:
main_data = pd.read_pickle("OR_Data_First_scrape.pickle")
main_data['PHYSICAL STATE'].value_counts()

PHYSICAL STATE
WA                      3150
OR                      2711
Kansas                    90
Louisiana                 69
Maryland                  52
Michigan                  50
Virginia                  50
Tennessee                 50
North Carolina            50
Mississippi               50
Massachusetts             49
South Carolina            49
Texas                     48
Georgia                   46
Missouri                  30
Delaware                  30
Oklahoma                  30
Pennsylvania Indiana      30
Alabama                   30
Ohio                      29
Arizona                   27
Washington                 7
Name: count, dtype: int64

In [5]:
import re
def extract_information(elements_list):
    extracted_data  = {}
    all_patterns = {"Reviews":r'^\d\.\d\(\d{1,3}(?:,\d{3})*\)$',
                    "BucinessType": r'\b(contractor|service|company|cleaning|repair|roofing|plumber|electrician|landscaping|heating|cooling|HVAC|construction)\b',
                    "ContactNo": r'\+\d{1,3}[\s-]?\d{3}[\s-]?\d{3}[\s-]?\d{4}',
                    "Address":r"\b(usa|united states)\b",
                    }
    for text in elements_list:
        for name,pattern in all_patterns.items():
            if isinstance(text, str) and re.search(pattern,text):
                extracted_data[name] = text

    return extracted_data
                
def extract_lat_long(url):
    
    match = re.search(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)", url)
    if match:
        lat, lng = match.groups()
    
    else:
        lat, lng = None,None
    return [lat,lng]

from urllib.parse import unquote

def extract_place_id(url):
    match = re.search(r"!16s(%2F[^!]+)", url)
    if match:
        place_id = unquote(match.group(1))
    else:
        place_id = None
    return place_id

In [6]:
def clean_attributes(main_data):
    main_data["CompanyName"] = main_data['all_text_first_page'].apply(lambda x:x.split("\n")[0])
    main_data["all_text_first_page_clean"] = main_data['all_text_first_page'].apply(lambda x:extract_information(x.split("\n")))

    main_data['LatLong']= main_data['google_map_link'].apply(lambda x:extract_lat_long(x))
    main_data['PlaceID']= main_data['google_map_link'].apply(lambda x:extract_place_id(x))
    cols = pd.json_normalize(main_data['all_text_first_page_clean']).columns
    main_data[cols] = pd.json_normalize(main_data['all_text_first_page_clean'])
    main_data[['Lat','Long']] = main_data['LatLong'].apply(pd.Series)
    return main_data

In [ ]:
final_clean = main_data[main_data['google_map_link'].notna()]
final_dump = clean_attributes(final_clean)
final_dump[['Rating_Indivisual','TotalReviews']] = final_dump['Reviews'].str.split("(").apply(pd.Series)
final_dump[['CompanyName','Lat', 'Long',  'TotalReviews','Rating_Indivisual',
'PlaceID', 'Reviews','BucinessType', 'ContactNo','DELIVERY ZIPCODE', 'PHYSICAL CITY', 'PHYSICAL STATE',
 'google_map_link']].to_excel("Data_v2.xlsx")


#### Website and Stars Details

In [ ]:
import json

list_of_files = os.listdir("Data/OutputData/FinalScrape/temp_results4")
dfs = []
for ele in list_of_files:
    with open(f"Data/OutputData/FinalScrape/temp_results4/{ele}",'r') as d:
        df = json.load(d)
        final_df = pd.DataFrame(df)
        dfs.append(final_df)

fi_data = pd.concat(dfs)
fi_data_df = fi_data.reset_index(drop=True)

def extract_details(elements):
    main_dict = {"Address":None,"Website":None,"Phone":None,"Plus code":None}
    if isinstance(elements,list):
        for ele in elements:
            if ele :
                if "Address" in ele:
                    main_dict['Address'] = ele.split(":")[-1]
                if "Website" in ele:
                    main_dict['Website'] = ele.split(":")[-1]
                if "Phone" in ele:
                    main_dict['Phone'] = ele.split(":")[-1]
                if "Plus" in ele:
                    main_dict['Plus code'] = ele.split(":")[-1]
        return main_dict
    else:
        return main_dict

fi_data_df = fi_data_df[fi_data_df['google_map_link'].notna()]
fi_data_df = clean_attributes(fi_data_df)
fi_data_df[['Rating_Indivisual','TotalReviews']] = fi_data_df['Reviews'].str.split("(").apply(pd.Series)

fi_data_df["ElementsData"] = fi_data_df['all_overview'].apply(lambda x:extract_details(x))
fi_data_df[["Address","Website","Phone","Plus code"]] = pd.json_normalize(fi_data_df['ElementsData'])
fi_data_df[['5Star','4Star','3Star','2Star','1Star']] = pd.json_normalize(fi_data_df['start_dist'].apply(lambda x:x.get('aria-label')))

In [26]:
fi_data_df.columns

Index(['all_overview', 'start_dist', 'search_google', 'all_text_first_page',
       'google_map_link', 'DELIVERY ZIPCODE', 'PHYSICAL CITY',
       'PHYSICAL STATE', 'error', 'CompanyName', 'all_text_first_page_clean',
       'LatLong', 'PlaceID', 'Reviews', 'BucinessType', 'ContactNo', 'Lat',
       'Long', 'Rating_Indivisual', 'TotalReviews', 'ElementsData', 'Address',
       'Website', 'Phone', 'Plus code', '5Star', '4Star', '3Star', '2Star',
       '1Star'],
      dtype='object')

In [29]:
cols = ['CompanyName', 'LatLong', 'PlaceID', 'Reviews', 'BucinessType', 'ContactNo', 'Lat',
'Long', 'Rating_Indivisual', 'TotalReviews', 'ElementsData', 'Address',
'Website', 'Phone', 'Plus code', '5Star', '4Star', '3Star', '2Star',
'1Star','google_map_link', 'DELIVERY ZIPCODE', 'PHYSICAL CITY','PHYSICAL STATE']

fi_data_df[cols].to_clipboard()

In [ ]:
["Washington ","Oregon ","Pennsylvania Indiana","Georgia","South Carolina","North Carolina","Virginia",
"Tennessee","Massachusetts","Maryland","Delaware", "Texas","Michigan","Ohio","Missouri","Kansas","Oklahoma",
"Arizona","Louisiana","Alabama","Mississippi"]